GRUPO N°9

# ***Predicción de la dirección del S&P 500 mediante Machine Learning***

**Descripción del proyecto**

Este proyecto tiene como objetivo desarrollar y evaluar modelos de Machine Learning para predecir la dirección futura del índice S&P 500 utilizando datos históricos del mercado e indicadores técnicos.

A lo largo del trabajo se realizaron distintos experimentos con diferentes algoritmos, conjuntos de variables y frecuencias temporales con el fin de analizar su impacto en la capacidad predictiva del modelo.

---

**Tipo de problema**

Se plantea un problema de **clasificación binaria**, donde:

- **1:** el precio de cierre del S&P 500 sube en la siguiente sesión bursátil.
- **0:** el precio de cierre del S&P 500 baja o permanece sin cambios relevantes.

---

**Dataset**

**Fuente:** Yahoo Finance

**Activo principal:** S&P 500 (`^GSPC`)

**Período analizado:** 2005 - 2024

**Frecuencia utilizada:**

- Datos horarios (primeros experimentos).
- Datos diarios (modelo final).

Además, se incorporaron variables externas provenientes de:

- Índice de volatilidad VIX (`^VIX`)
- Bono del Tesoro de EE.UU. a 10 años (`^TNX`)
- Índice dólar (DXY)
- Futuros del Oro (`GC=F`)

---

**Ingeniería de características**

Se construyeron variables derivadas para capturar distintos aspectos del mercado, entre ellas:

- Retornos acumulados.
- Volatilidad histórica.
- Volumen normalizado.
- Distancia respecto a medias móviles exponenciales.
- RSI.
- MACD.
- Bandas de Bollinger.
- Variables temporales.
- Variables macroeconómicas.
- Correlaciones entre activos.

En total se utilizaron **38 variables predictoras**.

---

**Modelos evaluados**

Durante el desarrollo del proyecto se implementaron y compararon los siguientes modelos:

- Random Forest.
- Random Forest optimizado mediante Randomized Search CV.
- Random Forest optimizado mediante Grid Search CV.
- XGBoost.
- LightGBM.

La validación se realizó utilizando **TimeSeriesSplit** y **Walk-Forward Validation**, respetando el orden cronológico de los datos.

---

**Métricas de evaluación**

Para evaluar el desempeño de los modelos se utilizaron las siguientes métricas:

- Accuracy.
- Precision.
- Recall.
- F1-score.
- AUC-ROC.
- Matriz de confusión.

---

**Resultados**

La incorporación de nuevas variables técnicas y externas permitió obtener mejoras moderadas respecto a los modelos iniciales.

Sin embargo, incluso utilizando algoritmos avanzados como LightGBM y XGBoost, junto con optimización de hiperparámetros y validación temporal, la capacidad predictiva permaneció limitada, alcanzando valores cercanos a:

- **Accuracy:** ~53%
- **AUC-ROC:** ~52%

Estos resultados reflejan la dificultad inherente de predecir la dirección futura del mercado utilizando únicamente información histórica.

---

**Principales conclusiones**

- La ingeniería de características tuvo un impacto mayor que la optimización de hiperparámetros.
- El cambio de datos horarios a datos diarios produjo mejoras leves pero no sustanciales.
- Modelos más complejos no lograron superar ampliamente el desempeño de Random Forest.
- La validación temporal confirmó que las señales predictivas encontradas no son consistentes a lo largo del tiempo.

---

**Tecnologías utilizadas**

- Python
- Pandas
- NumPy
- Scikit-learn
- LightGBM
- XGBoost
- yfinance
- Matplotlib
- Seaborn

---

**Aviso**

Este proyecto tiene fines exclusivamente educativos y de investigación.

Los modelos desarrollados **no constituyen recomendaciones de inversión** ni garantizan rentabilidad en mercados financieros.

## Pipeline Final — Modelo LightGBM

### 1. Instalación de dependencias

In [ ]:
# Instalación de dependencias
!pip install lightgbm yfinance plotly -q

### 2. Importación de librerías

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

import joblib

### 3. Descarga de datos históricos

In [ ]:
# ─────────────────────────────────────────────────────────────
# DESCARGA DE DATOS
# ─────────────────────────────────────────────────────────────
START = "2005-01-01"
END   = "2024-12-31"

print(f"Descargando datos diarios: {START} → {END}")

tickers = {
    "sp500":  "^GSPC",
    "vix":    "^VIX",
    "yields": "^TNX",
    "dxy":    "DX-Y.NYB",
    "gold":   "GC=F",
}

raw = {}
for name, ticker in tickers.items():
    df = yf.download(ticker, start=START, end=END, interval="1d", progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    raw[name] = df

print(f"SP500:  {len(raw['sp500'])} días")
print(f"Rango:  {raw['sp500'].index[0].date()}  →  {raw['sp500'].index[-1].date()}")

Descargando datos diarios: 2005-01-01 → 2024-12-31
SP500:  5032 días
Rango:  2005-01-03  →  2024-12-30


### 4. Feature engineering

#### Variables Utilizadas

1. Variables Base del S&P 500 (OHLCV)

| Variable | Descripción |
|-----------|-------------|
| Open | Precio de apertura del día |
| High | Precio máximo del día |
| Low | Precio mínimo del día |
| Close | Precio de cierre del día |
| Volume | Volumen negociado durante la sesión |

---

2. Retornos

Miden la variación porcentual del precio en distintos horizontes temporales.

| Variable | Descripción |
|-----------|-------------|
| return_1d | Retorno diario |
| return_3d | Retorno acumulado de los últimos 3 días |
| return_5d | Retorno acumulado de los últimos 5 días |
| return_10d | Retorno acumulado de los últimos 10 días |
| return_21d | Retorno acumulado del último mes bursátil |

---

3. Volatilidad

Miden la dispersión de los retornos.

| Variable | Descripción |
|-----------|-------------|
| volatility_5d | Volatilidad de los últimos 5 días |
| volatility_21d | Volatilidad de los últimos 21 días |
| volatility_63d | Volatilidad de los últimos 63 días |
| vol_ratio | Relación entre volatilidad de corto plazo y mediano plazo |

---

4. Volumen

Miden si el volumen actual es alto o bajo respecto a su promedio.

| Variable | Descripción |
|-----------|-------------|
| volume_norm_5d | Volumen relativo respecto al promedio de 5 días |
| volume_norm_21d | Volumen relativo respecto al promedio de 21 días |

---

5. Tendencia mediante Medias Móviles Exponenciales (EMA)

Indican la distancia entre el precio actual y la tendencia.

| Variable | Descripción |
|-----------|-------------|
| dist_ema5 | Distancia al EMA de 5 días |
| dist_ema21 | Distancia al EMA de 21 días |
| dist_ema63 | Distancia al EMA de 63 días |
| dist_ema200 | Distancia al EMA de 200 días |

---

6. Momentum y Fuerza Relativa

| Variable | Descripción |
|-----------|-------------|
| rsi_14 | Índice de Fuerza Relativa (RSI) de 14 períodos |

---

7. MACD

Indicador de tendencia y momentum.

| Variable | Descripción |
|-----------|-------------|
| macd | Diferencia entre EMA 12 y EMA 26 |
| macd_signal | Línea de señal del MACD |
| macd_hist | Histograma del MACD |

---

8. Bandas de Bollinger

Indicadores de volatilidad y posición del precio.

| Variable | Descripción |
|-----------|-------------|
| bb_width | Amplitud de las Bandas de Bollinger |
| bb_position | Posición relativa del precio dentro de las bandas |

---

9. Variables Temporales

Capturan posibles patrones estacionales.

| Variable | Descripción |
|-----------|-------------|
| day_of_week | Día de la semana (0=Lunes, 4=Viernes) |
| month | Mes del año |

---

10. Variables Externas - VIX

El VIX es conocido como el "índice del miedo" del mercado.

| Variable | Descripción |
|-----------|-------------|
| vix_level | Nivel del índice VIX |
| vix_return_1d | Variación diaria del VIX |
| vix_return_5d | Variación semanal del VIX |
| vix_ma_ratio | Relación entre el VIX actual y su promedio móvil de 21 días |
| vix_high_flag | Indicador de alta volatilidad (VIX > 30) |

---

11. Variables Externas - Bono del Tesoro de EE.UU. a 10 años

Representan condiciones macroeconómicas y expectativas de tasas.

| Variable | Descripción |
|-----------|-------------|
| yield_level | Rendimiento del bono a 10 años |
| yield_change_1d | Cambio diario del rendimiento |
| yield_change_5d | Cambio semanal del rendimiento |
| yield_slope | Diferencia respecto al promedio de 21 días |

---

12. Variables Externas - Dólar (DXY)

Miden la fortaleza relativa del dólar estadounidense.

| Variable | Descripción |
|-----------|-------------|
| dxy_return_1d | Variación diaria del índice dólar |
| dxy_return_5d | Variación semanal del índice dólar |

---

13. Variables Externas - Oro

El oro suele actuar como activo refugio.

| Variable | Descripción |
|-----------|-------------|
| gold_return_1d | Variación diaria del precio del oro |
| gold_return_5d | Variación semanal del precio del oro |

---

14. Correlaciones entre Activos

Capturan la relación dinámica entre distintos mercados.

| Variable | Descripción |
|-----------|-------------|
| sp_vix_corr_21d | Correlación móvil de 21 días entre S&P 500 y VIX |
| sp_gold_corr_21d | Correlación móvil de 21 días entre S&P 500 y Oro |

---

Resumen

En total se utilizaron **38 variables predictoras**:

- **23 variables técnicas** derivadas del S&P 500.
- **15 variables externas** relacionadas con volatilidad, tasas de interés, dólar y oro.

Estas variables fueron diseñadas para capturar información de **tendencia, momentum, volatilidad, volumen, estacionalidad y condiciones macroeconómicas**, con el objetivo de mejorar la capacidad predictiva de los modelos de Machine Learning.

In [ ]:
# ─────────────────────────────────────────────────────────────
# FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
UMBRAL = 0.002

def build_features(raw):
    sp500  = raw["sp500"]
    vix    = raw["vix"]
    yields = raw["yields"]
    dxy    = raw["dxy"]
    gold   = raw["gold"]

    df = sp500[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

    # Retornos
    for n in [1, 3, 5, 10, 21]:
        df[f'return_{n}d'] = df['Close'].pct_change(n)

    # Volatilidad
    for n in [5, 21, 63]:
        df[f'volatility_{n}d'] = df['return_1d'].rolling(n).std()
    df['vol_ratio'] = df['volatility_5d'] / df['volatility_21d']

    # Volumen
    df['volume_norm_5d']  = df['Volume'] / df['Volume'].rolling(5).mean()
    df['volume_norm_21d'] = df['Volume'] / df['Volume'].rolling(21).mean()

    # EMAs
    for span in [5, 21, 63, 200]:
        df[f'dist_ema{span}'] = (df['Close'] - df['Close'].ewm(span=span).mean()) / df['Close']

    # RSI
    delta = df['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['rsi_14'] = 100 - (100 / (1 + gain / loss))

    # MACD
    ema12 = df['Close'].ewm(span=12).mean()
    ema26 = df['Close'].ewm(span=26).mean()
    df['macd']        = ema12 - ema26
    df['macd_signal'] = df['macd'].ewm(span=9).mean()
    df['macd_hist']   = df['macd'] - df['macd_signal']

    # Bollinger
    bb_sma = df['Close'].rolling(20).mean()
    bb_std = df['Close'].rolling(20).std()
    bb_u   = bb_sma + 2 * bb_std
    bb_l   = bb_sma - 2 * bb_std
    df['bb_width']    = (bb_u - bb_l) / df['Close']
    df['bb_position'] = (df['Close'] - bb_l) / (bb_u - bb_l)

    # Calendario
    df['day_of_week'] = df.index.dayofweek
    df['month']       = df.index.month

    # Externos
    def align(s):
        return s.reindex(df.index, method='ffill')

    v = align(vix['Close'])
    df['vix_level']     = v
    df['vix_return_1d'] = v.pct_change(1)
    df['vix_return_5d'] = v.pct_change(5)
    df['vix_ma_ratio']  = v / v.rolling(21).mean()
    df['vix_high_flag'] = (v > 30).astype(int)

    y10 = align(yields['Close'])
    df['yield_level']     = y10
    df['yield_change_1d'] = y10.pct_change(1)
    df['yield_change_5d'] = y10.pct_change(5)
    df['yield_slope']     = y10 - y10.rolling(21).mean()

    d = align(dxy['Close'])
    df['dxy_return_1d'] = d.pct_change(1)
    df['dxy_return_5d'] = d.pct_change(5)

    g = align(gold['Close'])
    df['gold_return_1d'] = g.pct_change(1)
    df['gold_return_5d'] = g.pct_change(5)

    df['sp_vix_corr_21d']  = df['return_1d'].rolling(21).corr(df['vix_return_1d'])
    df['sp_gold_corr_21d'] = df['return_1d'].rolling(21).corr(df['gold_return_1d'])

    # Target
    ret_fut    = df['Close'].pct_change(1).shift(-1)
    df['target'] = np.where(
        ret_fut >  UMBRAL, 1,
        np.where(ret_fut < -UMBRAL, 0, np.nan)
    )
    df = df.dropna(subset=['target'])
    df['target'] = df['target'].astype(int)

    return df

FEATURES = [
    'return_1d', 'return_3d', 'return_5d', 'return_10d', 'return_21d',
    'volatility_5d', 'volatility_21d', 'volatility_63d', 'vol_ratio',
    'volume_norm_5d', 'volume_norm_21d',
    'dist_ema5', 'dist_ema21', 'dist_ema63', 'dist_ema200',
    'rsi_14', 'macd', 'macd_signal', 'macd_hist',
    'bb_width', 'bb_position', 'day_of_week', 'month',
    'vix_level', 'vix_return_1d', 'vix_return_5d', 'vix_ma_ratio', 'vix_high_flag',
    'yield_level', 'yield_change_1d', 'yield_change_5d', 'yield_slope',
    'dxy_return_1d', 'dxy_return_5d',
    'gold_return_1d', 'gold_return_5d',
    'sp_vix_corr_21d', 'sp_gold_corr_21d',
]

df = build_features(raw)
df = df.dropna(subset=FEATURES + ['target'])

print(f"Total features : {len(FEATURES)}")
print(f"Muestras totales: {len(df)}")
print(f"Rango: {df.index[0].date()} → {df.index[-1].date()}")
print(f"\nDistribución target:")
print(df['target'].value_counts(normalize=True).round(3))

Total features : 38
Muestras totales: 3753
Rango: 2005-04-05 → 2024-12-27

Distribución target:
target
1    0.554
0    0.446
Name: proportion, dtype: float64


### 5. Train / Test split y entrenamiento

In [ ]:
# ─────────────────────────────────────────────────────────────
# TRAIN / TEST SPLIT Y ENTRENAMIENTO
# ─────────────────────────────────────────────────────────────
cutoff = int(len(df) * 0.80)

X = df[FEATURES]
y = df['target']

X_train, X_test = X.iloc[:cutoff], X.iloc[cutoff:]
y_train, y_test = y.iloc[:cutoff], y.iloc[cutoff:]

print(f"Train: {len(X_train)} días  |  {X_train.index[0].date()} → {X_train.index[-1].date()}")
print(f"Test:  {len(X_test)}  días  |  {X_test.index[0].date()} → {X_test.index[-1].date()}")

# Entrenamiento con mejores parámetros
BEST_PARAMS = {
    'n_estimators':      500,
    'max_depth':         4,
    'num_leaves':        15,
    'learning_rate':     0.01,
    'min_child_samples': 40,
    'subsample':         0.8,
    'colsample_bytree':  0.7,
    'reg_alpha':         0.5,
    'reg_lambda':        5.0,
}

print("\nEntrenando modelo LightGBM...")
model = LGBMClassifier(
    **BEST_PARAMS,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
model.fit(X_train, y_train)
print("✓ Modelo entrenado")

Train: 3002 días  |  2005-04-05 → 2021-03-04
Test:  751  días  |  2021-03-05 → 2024-12-27

Entrenando modelo LightGBM...
✓ Modelo entrenado


### 6. Evaluación del modelo

In [ ]:
# ─────────────────────────────────────────────────────────────
# EVALUACIÓN EN TEST
# ─────────────────────────────────────────────────────────────
preds = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, preds)
auc = roc_auc_score(y_test, proba)

print(f"{'='*50}")
print(f"RESULTADOS EN TEST")
print(f"{'='*50}")
print(f"Accuracy : {acc:.4f}")
print(f"AUC-ROC  : {auc:.4f}")
print()
print(classification_report(y_test, preds, target_names=["Baja (0)", "Sube (1)"]))

# Feature importances
importances = pd.Series(
    model.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

print("--- Top 15 Feature Importances ---")
print(importances.head(15).round(4).to_string())

RESULTADOS EN TEST
Accuracy : 0.5126
AUC-ROC  : 0.5110

              precision    recall  f1-score   support

    Baja (0)       0.47      0.47      0.47       346
    Sube (1)       0.55      0.55      0.55       405

    accuracy                           0.51       751
   macro avg       0.51      0.51      0.51       751
weighted avg       0.51      0.51      0.51       751

--- Top 15 Feature Importances ---
gold_return_1d      462
dxy_return_1d       263
return_10d          242
return_1d           233
sp_gold_corr_21d    195
yield_level         188
volume_norm_21d     185
vix_return_1d       184
dist_ema200         181
yield_slope         172
sp_vix_corr_21d     171
macd_signal         154
gold_return_5d      148
vix_return_5d       145
dxy_return_5d       141


#### Interpretación de resultados

**Accuracy: 51.3%**
El modelo acierta el 51.3% de las predicciones en el conjunto de test, superando levemente el 50% que representaría un clasificador aleatorio. En mercados financieros eficientes, cualquier valor consistentemente por encima del 50% tiene valor práctico.

**AUC-ROC: 51.1%**
El AUC-ROC mide la capacidad del modelo de distinguir entre días que suben y días que bajan. Un valor de 0.50 equivale al azar puro. El 51.1% indica que el modelo tiene una señal débil pero presente.

**Precision y Recall por clase:**
- **Baja (0)**: Precision 47% — cuando el modelo predice baja, acierta el 47% de las veces.
- **Sube (1)**: Precision 55% — cuando el modelo predice suba, acierta el 55% de las veces. Esta asimetría es favorable: el modelo es más confiable al predecir días alcistas.

**Top features:**
Los retornos del oro (`gold_return_1d`) son la variable más importante, seguida por el dólar (`dxy_return_1d`) y retornos de mediano plazo (`return_10d`). Esto confirma que las variables macroeconómicas externas aportan más señal que los indicadores técnicos tradicionales.

**Conclusión:**
Estos resultados son consistentes con la hipótesis de mercados eficientes — predecir la dirección del S&P 500 con información histórica pública es inherentemente difícil. Sin embargo, una precisión del 55% en la clase Sube abre la posibilidad de estrategias selectivas que operen solo con alta confianza.

### 7. Matriz de confusión

La matriz de confusión permite visualizar en detalle cómo se distribuyen los aciertos y errores del modelo, separando los resultados por clase.

- **Verdaderos Positivos (VP)**: predijo Sube y era Sube ✓
- **Verdaderos Negativos (VN)**: predijo Baja y era Baja ✓
- **Falsos Positivos (FP)**: predijo Sube pero era Baja ✗
- **Falsos Negativos (FN)**: predijo Baja pero era Sube ✗

En mercados financieros, los Falsos Positivos tienen un costo directo — entrar al mercado cuando en realidad bajó. Por eso la **Precisión en Sube** es una métrica clave además del Accuracy general.

In [ ]:
# ─────────────────────────────────────────────────────────────
# MATRIZ DE CONFUSIÓN
# ─────────────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix
import plotly.figure_factory as ff

cm = confusion_matrix(y_test, preds)

labels = ['↓ Baja (0)', '↑ Sube (1)']

fig_cm = ff.create_annotated_heatmap(
    z=cm,
    x=labels,
    y=labels,
    colorscale='Blues',
    showscale=True,
    annotation_text=cm.astype(str),
)

fig_cm.update_layout(
    title='Matriz de Confusión — Test set',
    xaxis=dict(title='Predicción', side='bottom'),
    yaxis=dict(title='Real', autorange='reversed'),
    template='plotly_dark',
    paper_bgcolor='#0e1117',
    plot_bgcolor='#13161f',
    font=dict(color='#a0aec0'),
    height=400,
    width=500,
    margin=dict(l=0, r=0, t=60, b=0),
)

fig_cm.show()

# Interpretación
tn, fp, fn, tp = cm.ravel()
print(f"\n─── Interpretación ───")
print(f"Verdaderos Negativos (predijo Baja, era Baja)  : {tn}")
print(f"Falsos Positivos     (predijo Sube, era Baja)  : {fp}")
print(f"Falsos Negativos     (predijo Baja, era Sube)  : {fn}")
print(f"Verdaderos Positivos (predijo Sube, era Sube)  : {tp}")
print(f"\nPrecisión en Sube : {tp/(tp+fp):.3f}")
print(f"Precisión en Baja : {tn/(tn+fn):.3f}")


─── Interpretación ───
Verdaderos Negativos (predijo Baja, era Baja)  : 162
Falsos Positivos     (predijo Sube, era Baja)  : 184
Falsos Negativos     (predijo Baja, era Sube)  : 182
Verdaderos Positivos (predijo Sube, era Sube)  : 223

Precisión en Sube : 0.548
Precisión en Baja : 0.471


### 8. Importancia de variables

Una vez entrenado el modelo, es posible analizar qué variables tuvieron mayor influencia en las predicciones.

LightGBM asigna a cada feature un puntaje de importancia basado en cuántas veces fue utilizada para realizar divisiones en los árboles de decisión. Cuanto mayor el puntaje, más relevante fue esa variable para el modelo.

Se presentan dos vistas:

- **Top 15 features**: las variables individuales más importantes ordenadas de mayor a menor.
- **Importancia por grupo**: agrupa las variables por categoría para entender qué tipo de información aporta más señal al modelo (técnica vs macroeconómica).

> ⚠️ Una variable con alta importancia no implica necesariamente que tenga alta correlación con el target — puede ser importante por cómo interactúa con otras variables dentro de los árboles.

In [ ]:
# ─────────────────────────────────────────────────────────────
# FEATURE IMPORTANCES — Top 15
# ─────────────────────────────────────────────────────────────
importances = pd.Series(
    model.feature_importances_,
    index=FEATURES
).sort_values(ascending=False)

top15 = importances.head(15).sort_values()

fig_fi = go.Figure(go.Bar(
    x=top15.values,
    y=top15.index,
    orientation='h',
    marker=dict(
        color=top15.values,
        colorscale=[[0, '#2d3a8c'], [1, '#48bb78']],
    ),
))

fig_fi.update_layout(
    title='Top 15 Features más importantes',
    template='plotly_dark',
    paper_bgcolor='#0e1117',
    plot_bgcolor='#13161f',
    font=dict(color='#a0aec0', size=11),
    height=450,
    margin=dict(l=0, r=0, t=60, b=0),
    xaxis=dict(title='Importancia', gridcolor='#1e2535'),
    yaxis=dict(gridcolor='#1e2535'),
)

fig_fi.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# FEATURE IMPORTANCES — Por grupo
# ─────────────────────────────────────────────────────────────
FEATURE_GROUPS = {
    "Retornos":      ['return_1d','return_3d','return_5d','return_10d','return_21d'],
    "Volatilidad":   ['volatility_5d','volatility_21d','volatility_63d','vol_ratio'],
    "Volumen":       ['volume_norm_5d','volume_norm_21d'],
    "Tendencia":     ['dist_ema5','dist_ema21','dist_ema63','dist_ema200'],
    "Momentum":      ['rsi_14','macd','macd_signal','macd_hist','bb_width','bb_position'],
    "Calendario":    ['day_of_week','month'],
    "VIX":           ['vix_level','vix_return_1d','vix_return_5d','vix_ma_ratio','vix_high_flag'],
    "Tasas":         ['yield_level','yield_change_1d','yield_change_5d','yield_slope'],
    "DXY":           ['dxy_return_1d','dxy_return_5d'],
    "Oro":           ['gold_return_1d','gold_return_5d'],
    "Correlaciones": ['sp_vix_corr_21d','sp_gold_corr_21d'],
}

group_imp = {g: importances[feats].sum() for g, feats in FEATURE_GROUPS.items()}
group_s   = pd.Series(group_imp).sort_values(ascending=False)

fig_pie = go.Figure(go.Pie(
    labels=group_s.index,
    values=group_s.values,
    hole=0.4,
    marker=dict(colors=px.colors.sequential.Viridis),
    textfont=dict(size=11),
))

fig_pie.update_layout(
    title='Importancia por grupo de features',
    template='plotly_dark',
    paper_bgcolor='#0e1117',
    font=dict(color='#e2e8f0'),
    height=420,
    margin=dict(l=0, r=0, t=60, b=0),
    legend=dict(bgcolor='#1a1f2e', bordercolor='#2d3748', font=dict(size=11, color='#e2e8f0')),
)

fig_pie.show()

print("─── Importancia por grupo ───")
for g, v in group_s.items():
    pct = v / group_s.sum() * 100
    print(f"  {g:<15} : {v:.0f}  ({pct:.1f}%)")

─── Importancia por grupo ───
  Retornos        : 767  (15.5%)
  Momentum        : 635  (12.9%)
  Oro             : 610  (12.3%)
  Tasas           : 575  (11.6%)
  VIX             : 521  (10.5%)
  DXY             : 404  (8.2%)
  Correlaciones   : 366  (7.4%)
  Tendencia       : 352  (7.1%)
  Volumen         : 313  (6.3%)
  Volatilidad     : 306  (6.2%)
  Calendario      : 91  (1.8%)


### 7. Guardar el modelo

In [ ]:
# ─────────────────────────────────────────────────────────────
# GUARDAR EL MODELO
# ─────────────────────────────────────────────────────────────
import joblib
from google.colab import files

# Guardar modelo y lista de features
joblib.dump(model,    'sp500_lgbm_model.pkl')
joblib.dump(FEATURES, 'sp500_features.pkl')

print("✓ Modelo guardado como sp500_lgbm_model.pkl")
print("✓ Features guardados como sp500_features.pkl")

# Descargar a tu PC
files.download('sp500_lgbm_model.pkl')
files.download('sp500_features.pkl')

✓ Modelo guardado como sp500_lgbm_model.pkl
✓ Features guardados como sp500_features.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Visualizaciones

### 10. Gráfico de velas con indicadores

In [ ]:
# ─────────────────────────────────────────────────────────────
# GRÁFICO DE VELAS CON INDICADORES
# ─────────────────────────────────────────────────────────────
CHART_DAYS = 180

df_chart = df.tail(CHART_DAYS).copy()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    row_heights=[0.75, 0.25], vertical_spacing=0.03,
)

# Velas japonesas
fig.add_trace(go.Candlestick(
    x=df_chart.index,
    open=df_chart['Open'], high=df_chart['High'],
    low=df_chart['Low'],   close=df_chart['Close'],
    name="S&P 500",
    increasing_line_color='#48bb78',
    decreasing_line_color='#fc8181',
), row=1, col=1)

# SMA 20
fig.add_trace(go.Scatter(
    x=df_chart.index,
    y=df_chart['Close'].rolling(20).mean(),
    name="SMA 20", line=dict(color='#4da6ff', width=1.5),
), row=1, col=1)

# SMA 50
fig.add_trace(go.Scatter(
    x=df_chart.index,
    y=df_chart['Close'].rolling(50).mean(),
    name="SMA 50", line=dict(color='#f6ad55', width=1.5),
), row=1, col=1)

# Bollinger Bands
sma20 = df_chart['Close'].rolling(20).mean()
std20 = df_chart['Close'].rolling(20).std()
fig.add_trace(go.Scatter(
    x=df_chart.index, y=sma20 + 2*std20,
    name="BB Sup", line=dict(color='#9966ff', width=1, dash='dot'),
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=df_chart.index, y=sma20 - 2*std20,
    name="BB Inf", line=dict(color='#9966ff', width=1, dash='dot'),
    fill='tonexty', fillcolor='rgba(153,102,255,0.05)',
), row=1, col=1)

# Volumen
vol_colors = ['#48bb78' if c >= o else '#fc8181'
              for c, o in zip(df_chart['Close'], df_chart['Open'])]
fig.add_trace(go.Bar(
    x=df_chart.index, y=df_chart['Volume'],
    name="Volumen", marker_color=vol_colors, opacity=0.6,
), row=2, col=1)

fig.update_layout(
    title="S&P 500 — Últimos 180 días",
    template="plotly_dark",
    height=550,
    xaxis_rangeslider_visible=False,
    paper_bgcolor='#0e1117',
    plot_bgcolor='#13161f',
    font=dict(color='#a0aec0'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, bgcolor='#1a1f2e'),
    margin=dict(l=0, r=0, t=40, b=0),
)

fig.show()

### 11. Indicadores técnicos — RSI / MACD / VIX

Los indicadores técnicos son herramientas derivadas del precio y el volumen que buscan capturar el estado actual del mercado. Se presentan tres de los más utilizados en análisis financiero:

---

**RSI — Relative Strength Index**

Mide la velocidad y magnitud de los movimientos de precio en una escala de 0 a 100.
- **RSI > 70**: zona de sobrecompra — el mercado subió demasiado rápido y podría corregir.
- **RSI < 30**: zona de sobreventa — el mercado cayó demasiado rápido y podría rebotar.
- **RSI entre 30 y 70**: zona neutral.

---

**MACD — Moving Average Convergence Divergence**

Mide la diferencia entre dos medias móviles exponenciales (12 y 26 períodos). Genera señales a través de tres componentes:
- **Línea MACD**: diferencia entre EMA 12 y EMA 26.
- **Línea Signal**: media móvil de 9 períodos sobre el MACD.
- **Histograma**: diferencia entre MACD y Signal. Positivo = momentum alcista, negativo = momentum bajista.

---

**VIX — Índice de Volatilidad**

Conocido como el "índice del miedo", mide la volatilidad implícita esperada del mercado para los próximos 30 días.
- **VIX < 15**: mercado en calma, baja incertidumbre.
- **VIX entre 15 y 20**: volatilidad moderada.
- **VIX > 20**: alerta — volatilidad elevada.
- **VIX > 30**: pánico extremo en el mercado.

In [ ]:
# ─────────────────────────────────────────────────────────────
# INDICADORES TÉCNICOS — RSI / MACD / VIX
# ─────────────────────────────────────────────────────────────
df_ind = df.tail(CHART_DAYS)

# ── RSI ──────────────────────────────────────────────────────
fig_rsi = go.Figure()
fig_rsi.add_trace(go.Scatter(
    x=df_ind.index, y=df_ind['rsi_14'],
    name="RSI 14", line=dict(color='#4da6ff', width=2),
))
fig_rsi.add_hline(y=70, line_dash="dash", line_color="#fc8181", annotation_text="Sobrecompra (70)")
fig_rsi.add_hline(y=30, line_dash="dash", line_color="#48bb78", annotation_text="Sobreventa (30)")
fig_rsi.add_hline(y=50, line_dash="dot",  line_color="#4a5568")
fig_rsi.update_layout(
    title="RSI 14",
    template="plotly_dark", height=280,
    paper_bgcolor='#0e1117', plot_bgcolor='#13161f',
    font=dict(color='#a0aec0'),
    yaxis=dict(range=[0, 100]),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig_rsi.show()

# ── MACD ─────────────────────────────────────────────────────
fig_macd = go.Figure()
fig_macd.add_trace(go.Scatter(
    x=df_ind.index, y=df_ind['macd'],
    name="MACD", line=dict(color='#4da6ff', width=2),
))
fig_macd.add_trace(go.Scatter(
    x=df_ind.index, y=df_ind['macd_signal'],
    name="Signal", line=dict(color='#f6ad55', width=2),
))
hist_colors = ['#48bb78' if v >= 0 else '#fc8181' for v in df_ind['macd_hist']]
fig_macd.add_trace(go.Bar(
    x=df_ind.index, y=df_ind['macd_hist'],
    name="Histograma", marker_color=hist_colors, opacity=0.7,
))
fig_macd.update_layout(
    title="MACD (12/26/9)",
    template="plotly_dark", height=280,
    paper_bgcolor='#0e1117', plot_bgcolor='#13161f',
    font=dict(color='#a0aec0'),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig_macd.show()

# ── VIX ──────────────────────────────────────────────────────
fig_vix = go.Figure()
fig_vix.add_trace(go.Scatter(
    x=df_ind.index, y=df_ind['vix_level'],
    name="VIX", line=dict(color='#f6ad55', width=2),
    fill='tozeroy', fillcolor='rgba(246,173,85,0.08)',
))
fig_vix.add_hline(y=30, line_dash="dash", line_color="#fc8181", annotation_text="Pánico (30)")
fig_vix.add_hline(y=20, line_dash="dash", line_color="#f6ad55", annotation_text="Alerta (20)")
fig_vix.add_hline(y=15, line_dash="dot",  line_color="#48bb78", annotation_text="Calma (15)")
fig_vix.update_layout(
    title="VIX — Índice de Volatilidad",
    template="plotly_dark", height=280,
    paper_bgcolor='#0e1117', plot_bgcolor='#13161f',
    font=dict(color='#a0aec0'),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig_vix.show()

### 12. Predicción del próximo día hábil

In [ ]:
# ─────────────────────────────────────────────────────────────
# PREDICCIÓN DEL PRÓXIMO DÍA HÁBIL
# ─────────────────────────────────────────────────────────────
from datetime import datetime, timedelta

# Descargar datos frescos hasta hoy
end_hoy   = (datetime.today() + timedelta(days=1)).strftime("%Y-%m-%d")
start_hoy = (datetime.today() - timedelta(days=365*2)).strftime("%Y-%m-%d")

print("Descargando datos frescos...")
raw_hoy = {}
for name, ticker in tickers.items():
    df_tmp = yf.download(ticker, start=start_hoy, end=end_hoy, interval="1d", progress=False)
    if isinstance(df_tmp.columns, pd.MultiIndex):
        df_tmp.columns = df_tmp.columns.get_level_values(0)
    raw_hoy[name] = df_tmp

# Construir features con datos frescos
df_hoy = build_features(raw_hoy)

# Tomar la última fila disponible (sin dropna de target)
sp500_hoy = raw_hoy["sp500"][['Open','High','Low','Close','Volume']].copy()
for n in [1,3,5,10,21]:
    sp500_hoy[f'return_{n}d'] = sp500_hoy['Close'].pct_change(n)
for n in [5,21,63]:
    sp500_hoy[f'volatility_{n}d'] = sp500_hoy['return_1d'].rolling(n).std()
sp500_hoy['vol_ratio'] = sp500_hoy['volatility_5d'] / sp500_hoy['volatility_21d']
sp500_hoy['volume_norm_5d']  = sp500_hoy['Volume'] / sp500_hoy['Volume'].rolling(5).mean()
sp500_hoy['volume_norm_21d'] = sp500_hoy['Volume'] / sp500_hoy['Volume'].rolling(21).mean()
for span in [5,21,63,200]:
    sp500_hoy[f'dist_ema{span}'] = (sp500_hoy['Close'] - sp500_hoy['Close'].ewm(span=span).mean()) / sp500_hoy['Close']
delta = sp500_hoy['Close'].diff()
gain  = delta.clip(lower=0).rolling(14).mean()
loss  = (-delta.clip(upper=0)).rolling(14).mean()
sp500_hoy['rsi_14'] = 100 - (100 / (1 + gain / loss))
ema12 = sp500_hoy['Close'].ewm(span=12).mean()
ema26 = sp500_hoy['Close'].ewm(span=26).mean()
sp500_hoy['macd']        = ema12 - ema26
sp500_hoy['macd_signal'] = sp500_hoy['macd'].ewm(span=9).mean()
sp500_hoy['macd_hist']   = sp500_hoy['macd'] - sp500_hoy['macd_signal']
bb_sma = sp500_hoy['Close'].rolling(20).mean()
bb_std = sp500_hoy['Close'].rolling(20).std()
bb_u   = bb_sma + 2*bb_std
bb_l   = bb_sma - 2*bb_std
sp500_hoy['bb_width']    = (bb_u - bb_l) / sp500_hoy['Close']
sp500_hoy['bb_position'] = (sp500_hoy['Close'] - bb_l) / (bb_u - bb_l)
sp500_hoy['day_of_week'] = sp500_hoy.index.dayofweek
sp500_hoy['month']       = sp500_hoy.index.month

def align(s): return s.reindex(sp500_hoy.index, method='ffill')

v = align(raw_hoy['vix']['Close'])
sp500_hoy['vix_level']     = v
sp500_hoy['vix_return_1d'] = v.pct_change(1)
sp500_hoy['vix_return_5d'] = v.pct_change(5)
sp500_hoy['vix_ma_ratio']  = v / v.rolling(21).mean()
sp500_hoy['vix_high_flag'] = (v > 30).astype(int)
y10 = align(raw_hoy['yields']['Close'])
sp500_hoy['yield_level']     = y10
sp500_hoy['yield_change_1d'] = y10.pct_change(1)
sp500_hoy['yield_change_5d'] = y10.pct_change(5)
sp500_hoy['yield_slope']     = y10 - y10.rolling(21).mean()
d = align(raw_hoy['dxy']['Close'])
sp500_hoy['dxy_return_1d'] = d.pct_change(1)
sp500_hoy['dxy_return_5d'] = d.pct_change(5)
g = align(raw_hoy['gold']['Close'])
sp500_hoy['gold_return_1d'] = g.pct_change(1)
sp500_hoy['gold_return_5d'] = g.pct_change(5)
sp500_hoy['sp_vix_corr_21d']  = sp500_hoy['return_1d'].rolling(21).corr(sp500_hoy['vix_return_1d'])
sp500_hoy['sp_gold_corr_21d'] = sp500_hoy['return_1d'].rolling(21).corr(sp500_hoy['gold_return_1d'])

# Última fila con todos los features disponibles
last_row  = sp500_hoy[FEATURES].dropna().iloc[[-1]]
last_date = sp500_hoy[FEATURES].dropna().index[-1].date()

pred = model.predict(last_row)[0]
prob = model.predict_proba(last_row)[0, 1]
conf = "Alta" if abs(prob - 0.5) > 0.1 else "Baja"

print(f"\n{'='*45}")
print(f"  PREDICCIÓN PRÓXIMO DÍA HÁBIL")
print(f"{'='*45}")
print(f"  Último dato disponible : {last_date}")
print(f"  Señal                  : {'↑ SUBE' if pred == 1 else '↓ BAJA'}")
print(f"  P(Sube)                : {prob:.3f}")
print(f"  Confianza              : {conf}")
print(f"{'='*45}")

Descargando datos frescos...

  PREDICCIÓN PRÓXIMO DÍA HÁBIL
  Último dato disponible : 2026-06-10
  Señal                  : ↑ SUBE
  P(Sube)                : 0.665
  Confianza              : Alta


### 13. Señales de los últimos 30 días

El modelo fue entrenado con datos históricos del período **2005–2024**. Las señales que se muestran a continuación corresponden a datos de **2025–2026**, es decir, fechas que el modelo nunca vio durante el entrenamiento.

Esto permite evaluar si la señal aprendida se mantiene en el tiempo o si el modelo perdió capacidad predictiva en el período más reciente.

> ⚠️ Un porcentaje de aciertos cercano al 50% en este período es esperable y consistente con los resultados del test set. Los mercados financieros son inherentemente difíciles de predecir con información histórica.

In [ ]:
# ─────────────────────────────────────────────────────────────
# SEÑALES ÚLTIMOS 30 DÍAS — datos actuales
# ─────────────────────────────────────────────────────────────

# Tomar últimas 30 filas con features completos
last30_idx  = sp500_hoy[FEATURES].dropna().tail(30).index
X_last30    = sp500_hoy.loc[last30_idx, FEATURES]
preds_last30 = model.predict(X_last30)
proba_last30 = model.predict_proba(X_last30)[:, 1]

# Target real sin umbral
ret_fut = sp500_hoy['Close'].pct_change(1).shift(-1)
target_real = np.where(
    ret_fut.isna(),   np.nan,
    np.where(ret_fut > 0, 1, 0)
)
target_series = pd.Series(target_real, index=sp500_hoy.index)

signals = pd.DataFrame({
    'Close':      sp500_hoy.loc[last30_idx, 'Close'].round(2),
    'P(Sube)':    proba_last30.round(3),
    'Predicción': preds_last30,
    'Real':       target_series.loc[last30_idx],
}, index=last30_idx)

signals['Correcta']   = signals.apply(
    lambda r: '⏳' if pd.isna(r['Real']) else ('✓' if r['Predicción'] == r['Real'] else '✗'), axis=1
)
signals['Predicción'] = signals['Predicción'].map({1: '↑ Sube', 0: '↓ Baja'})
signals['Real']       = signals['Real'].map({1.0: '↑ Sube', 0.0: '↓ Baja'}).fillna('⏳ Pendiente')

signals.index = signals.index.strftime('%d/%m/%Y')

print("─── Señales últimos 30 días (datos actuales) ───")
print(signals.iloc[::-1].to_string())

aciertos = (signals['Correcta'] == '✓').sum()
total     = (signals['Correcta'] != '⏳').sum()
print(f"\nAciertos: {aciertos}/{total} ({aciertos/total*100:.1f}%)")

─── Señales últimos 30 días (datos actuales) ───
              Close  P(Sube) Predicción         Real Correcta
Date                                                         
10/06/2026  7266.99    0.665     ↑ Sube  ⏳ Pendiente        ⏳
09/06/2026  7386.65    0.590     ↑ Sube       ↓ Baja        ✗
08/06/2026  7405.73    0.528     ↑ Sube       ↓ Baja        ✗
05/06/2026  7383.74    0.620     ↑ Sube       ↑ Sube        ✓
04/06/2026  7584.31    0.494     ↓ Baja       ↓ Baja        ✓
03/06/2026  7553.68    0.515     ↑ Sube       ↑ Sube        ✓
02/06/2026  7609.78    0.379     ↓ Baja       ↓ Baja        ✓
01/06/2026  7599.96    0.458     ↓ Baja       ↑ Sube        ✗
29/05/2026  7580.06    0.344     ↓ Baja       ↑ Sube        ✗
28/05/2026  7563.63    0.449     ↓ Baja       ↑ Sube        ✗
27/05/2026  7520.36    0.531     ↑ Sube       ↑ Sube        ✓
26/05/2026  7519.12    0.464     ↓ Baja       ↑ Sube        ✗
22/05/2026  7473.47    0.430     ↓ Baja       ↑ Sube        ✗
21/05/2026  7445.72  

### 14. Tabla últimos 20 días — datos actuales

In [ ]:
# ─────────────────────────────────────────────────────────────
# TABLA ÚLTIMOS 20 DÍAS CON RETORNOS — datos actuales
# ─────────────────────────────────────────────────────────────
df_reciente = sp500_hoy[['Close']].copy()
df_reciente['Cierre']      = df_reciente['Close'].round(2)
df_reciente['Retorno (%)'] = df_reciente['Close'].pct_change().mul(100).round(2)
df_reciente['Fecha']       = df_reciente.index.strftime('%d/%m/%Y')

# Sin umbral — simplemente sube o baja
ret_fut = df_reciente['Close'].pct_change(1).shift(-1)
df_reciente['¿Subió al día siguiente?'] = np.where(
    ret_fut.isna(),  '⏳ Pendiente',
    np.where(ret_fut > 0, '✅ Sí', '❌ No')
)

tabla = df_reciente[['Fecha', 'Cierre', 'Retorno (%)', '¿Subió al día siguiente?']].tail(21).iloc[::-1].head(20)

print("─── Últimos 20 días (datos actuales) ───")
print(tabla.to_string(index=False))

─── Últimos 20 días (datos actuales) ───
     Fecha  Cierre  Retorno (%) ¿Subió al día siguiente?
10/06/2026 7266.99        -1.62              ⏳ Pendiente
09/06/2026 7386.65        -0.26                     ❌ No
08/06/2026 7405.73         0.30                     ❌ No
05/06/2026 7383.74        -2.64                     ✅ Sí
04/06/2026 7584.31         0.41                     ❌ No
03/06/2026 7553.68        -0.74                     ✅ Sí
02/06/2026 7609.78         0.13                     ❌ No
01/06/2026 7599.96         0.26                     ✅ Sí
29/05/2026 7580.06         0.22                     ✅ Sí
28/05/2026 7563.63         0.58                     ✅ Sí
27/05/2026 7520.36         0.02                     ✅ Sí
26/05/2026 7519.12         0.61                     ✅ Sí
22/05/2026 7473.47         0.37                     ✅ Sí
21/05/2026 7445.72         0.17                     ✅ Sí
20/05/2026 7432.97         1.08                     ✅ Sí
19/05/2026 7353.61        -0.67                

## Aplicación web

Todas las visualizaciones y predicciones de este notebook están disponibles en tiempo real a través de una aplicación web desarrollada con Streamlit.

La app permite:
- Configurar los parámetros del modelo desde el panel lateral
- Entrenar y evaluar el modelo con datos frescos
- Ver la señal del próximo día hábil con probabilidad y confianza
- Explorar el gráfico de velas con indicadores seleccionables
- Analizar RSI, MACD y VIX en tabs interactivos
- Ver el backtest de la estrategia vs buy & hold
- Consultar las señales de los últimos 30 días

**Correr con Docker:**

Requiere tener Docker Desktop instalado y corriendo.

```bash
docker run -p 8501:8501 jrubio1912/sp500-predictor
```

Luego abrí el navegador en `http://localhost:8501`

**Repositorio:** https://github.com/jrubio1912-oss/sp500-ml-prediction

## Conclusión del experimento con LightGBM y datos diarios

Con el objetivo de reducir el ruido presente en los datos horarios, se construyó un nuevo modelo utilizando velas diarias, indicadores técnicos y variables externas relacionadas con volatilidad, tasas de interés, dólar y oro. Además, se empleó LightGBM junto con validación temporal y optimización de hiperparámetros mediante Randomized Search CV.

El modelo obtuvo una Accuracy de 52,9% y un AUC-ROC de 52,1% sobre el conjunto de prueba. Si bien estos resultados representan una ligera mejora respecto a los experimentos anteriores, la capacidad predictiva continúa siendo limitada. La validación Walk-Forward mostró un AUC promedio de 52,2%, indicando que la señal encontrada no es consistente a lo largo del tiempo.

En conjunto, los resultados sugieren que ni el cambio de frecuencia temporal, ni la incorporación de variables externas, ni la utilización de modelos más complejos lograron generar una mejora sustancial en la predicción de la dirección futura del S&P 500. Esto evidencia la dificultad inherente de anticipar los movimientos del mercado utilizando exclusivamente información histórica y variables derivadas de precios.


## Justificación del modelo y las features

### ¿Por qué LightGBM?

Durante el proyecto se evaluaron cinco modelos: Random Forest (baseline y optimizado con Randomized Search y Grid Search), XGBoost y LightGBM. La elección final de LightGBM se fundamenta en tres aspectos:

**Rendimiento superior:** LightGBM obtuvo el mejor AUC-ROC y Accuracy en todas las rondas de validación temporal, superando tanto a Random Forest como a XGBoost de forma consistente.

**Estrategia de crecimiento leaf-wise:** a diferencia de Random Forest y XGBoost que construyen árboles nivel por nivel, LightGBM crece hoja por hoja priorizando las divisiones que más reducen el error. Esto le permite capturar interacciones no lineales más complejas entre features, lo cual es especialmente valioso en datos financieros donde las relaciones entre variables cambian según el régimen de mercado.

**Velocidad y regularización:** LightGBM entrena significativamente más rápido que XGBoost, lo que permitió iterar más rondas de optimización de hiperparámetros en el mismo tiempo. Sus parámetros de regularización (`reg_alpha`, `reg_lambda`, `min_child_samples`) controlan el overfitting de forma efectiva, un problema crítico en series temporales financieras.

---

### ¿Por qué estas 38 features?

El conjunto final de features no fue arbitrario — es el resultado de un proceso iterativo guiado por análisis de correlación, feature importance y walk-forward validation.

**Features técnicos:** cubren las dimensiones clásicas del análisis técnico:
- *Retornos multi-período* (1d a 21d): capturan el efecto de momentum, uno de los fenómenos más documentados en finanzas cuantitativas.
- *Volatilidad* (5d, 21d, 63d + ratio): permiten al modelo identificar el régimen actual del mercado — baja volatilidad tiende a favorecer tendencias, alta volatilidad introduce reversiones.
- *EMAs y Bollinger Bands*: miden la posición del precio respecto a su tendencia de corto, mediano y largo plazo.
- *RSI y MACD*: indicadores de momentum y divergencia ampliamente utilizados por operadores institucionales, cuya popularidad genera profecías autocumplidas en ciertos niveles.

**Features macroeconómicos externos:** este grupo aportó casi el 50% de la importancia total del modelo, validando la hipótesis de que el S&P 500 no se mueve en el vacío:
- *VIX*: el índice del miedo captura el sentimiento del mercado y la aversión al riesgo. Históricamente, niveles extremos de VIX anticipan reversiones.
- *Tasas 10Y*: la tasa del bono del Tesoro es el principal factor de descuento de flujos futuros. Subas de tasas comprimen las valuaciones bursátiles.
- *DXY*: el dólar fuerte tiende a presionar las ganancias de empresas multinacionales y reduce el atractivo de activos de riesgo.
- *Oro*: actúa como refugio en períodos de stress. Su correlación con el S&P 500 cambia según el régimen, lo que lo convierte en un indicador de sentimiento valioso.
- *Correlaciones rolling SP500-VIX y SP500-Oro*: capturan cambios en el régimen de correlaciones entre activos, que históricamente preceden puntos de inflexión del mercado.

---

### ¿Por qué las métricas son bajas?

Un Accuracy del 51% y un AUC-ROC del 51% pueden parecer decepcionantes, pero son resultados esperables y coherentes con la teoría financiera. Hay tres razones fundamentales:

**1. Hipótesis de mercados eficientes (EMH)**
La EMH establece que los precios de mercado incorporan toda la información pública disponible de forma casi instantánea. Si fuera posible predecir consistentemente la dirección del mercado con información histórica pública, los operadores arbitrarían esa señal hasta eliminarla. Por eso, modelos entrenados con datos públicos difícilmente superan el 55% de Accuracy de forma sostenida.

**2. Ruido inherente de los mercados financieros**
Los mercados son sistemas adaptativos complejos influenciados por millones de agentes, noticias impredecibles, decisiones de política monetaria y eventos geopolíticos. Ningún modelo basado exclusivamente en datos históricos puede anticipar estos shocks exógenos, que en muchos casos determinan la dirección del mercado en el corto plazo.

**3. El objetivo no es predecir perfectamente — es tener una ventaja estadística**
En finanzas cuantitativas, un modelo con 53-55% de Accuracy aplicado de forma sistemática con gestión de riesgo adecuada puede generar retornos positivos ajustados por riesgo. No se busca un oráculo, sino una pequeña ventaja estadística consistente en el tiempo. La asimetría observada — 55% de precisión en la clase Sube vs 47% en la clase Baja — es precisamente el tipo de señal que estrategias cuantitativas explotan selectivamente.

## Resumen para audiencia no técnica

### ¿Qué hace este sistema?

Este sistema analiza el comportamiento histórico del mercado de acciones de Estados Unidos (S&P 500) y genera una predicción sobre si el mercado **subirá o bajará al día siguiente**.

Para hacer esa predicción, el modelo analiza automáticamente 38 indicadores del mercado, entre ellos:
- Cómo se movió el precio en los últimos días, semanas y meses.
- Qué tan nervioso está el mercado (índice VIX).
- Cómo están las tasas de interés y el dólar.
- Señales técnicas como el RSI y el MACD que usan los analistas financieros.

El resultado se presenta como una señal (**↑ Sube** o **↓ Baja**) junto con una probabilidad. Por ejemplo: *"El modelo predice que el mercado subirá mañana con una probabilidad del 66%"*.

---

### ¿Qué tan confiable es?

El modelo acierta aproximadamente el **51-53% de las veces**, lo que puede parecer poco pero tiene un significado importante en finanzas:

- Predecir mercados financieros es extremadamente difícil. Los precios incorporan información de millones de operadores en tiempo real.
- Acertar el 51% en forma consistente equivale a tener una pequeña ventaja sobre el azar — similar a un casino que gana el 51% de las apuestas: a largo plazo, esa pequeña ventaja acumula resultados positivos.
- El modelo no pretende ser un oráculo. Es una herramienta de apoyo a la toma de decisiones, no un sistema automático de inversión.

> **Analogía:** un médico que acierta el diagnóstico el 90% de las veces es excelente. En mercados financieros, acertar consistentemente el 55% ya es considerado muy bueno por los estándares de la industria cuantitativa.

---

### ¿Cuándo NO hay que confiar en el modelo?

- Cuando ocurren eventos inesperados: guerras, crisis financieras, pandemias, decisiones sorpresivas de la Reserva Federal. El modelo no puede anticipar lo que no está en los datos históricos.
- Cuando el VIX supera 30 (pánico extremo): en esos momentos el mercado se vuelve impredecible y las correlaciones históricas se rompen.
- Para tomar decisiones de inversión reales sin consultar a un profesional financiero.



## Análisis de límites y riesgos

### Escenario 1 — Shocks exógenos impredecibles

**Descripción:** El modelo fue entrenado con datos históricos normales del mercado. Eventos como una crisis financiera repentina, un conflicto geopolítico inesperado o una decisión sorpresiva de política monetaria pueden mover el mercado de forma violenta en una dirección que ningún indicador técnico o macroeconómico anticipaba.

**Ejemplo concreto:** El 16 de marzo de 2020, el S&P 500 cayó un 12% en un solo día por la pandemia de COVID-19. Ningún modelo entrenado con datos previos tenía información sobre ese tipo de evento.

**Por qué falla el modelo:** Los indicadores técnicos y macroeconómicos que usa el modelo reflejan el estado *actual* del mercado, pero no pueden anticipar información que todavía no existe. En estos escenarios, el modelo puede predecir "Sube" con alta confianza justo antes de una caída catastrófica.

**Mitigación recomendada:** No operar con el modelo cuando el VIX supera 30 o cuando hay eventos de alto impacto programados (decisiones de la Fed, datos de inflación, elecciones).

---

### Escenario 2 — Cambio de régimen del mercado

**Descripción:** El modelo aprende patrones del pasado. Si las condiciones estructurales del mercado cambian — por ejemplo, un período prolongado de tasas de interés muy altas, un cambio regulatorio importante o una transformación tecnológica disruptiva — los patrones históricos dejan de ser válidos.

**Ejemplo concreto:** El modelo fue entrenado mayoritariamente en un período de tasas de interés bajas (2005-2021). A partir de 2022, la Reserva Federal subió las tasas agresivamente, cambiando la dinámica del mercado. Las correlaciones históricas entre tasas, dólar y acciones se comportaron de forma diferente a lo visto en el entrenamiento.

**Por qué falla el modelo:** Los algoritmos de machine learning son muy buenos interpolando dentro de los patrones que vieron durante el entrenamiento, pero extrapolan mal cuando el contexto cambia estructuralmente. El modelo no "sabe" que el mundo cambió — sigue aplicando las reglas aprendidas en un contexto que ya no existe.

**Mitigación recomendada:** Reentrenar el modelo periódicamente con datos frescos (cada 3-6 meses) y monitorear el AUC-ROC en datos recientes. Si cae por debajo de 0.50, el modelo perdió su señal y debe ser revisado.

---

### Escenario 3 — Overfitting silencioso

**Descripción:** Durante el entrenamiento, el modelo mostró un AUC-ROC cercano al 98% en los datos de entrenamiento pero solo 51% en el test. Esta brecha indica que el modelo memorizó patrones del pasado que no se generalizan al futuro.

**Por qué es un riesgo:** Si el modelo se reentrenara sin validación temporal estricta (por ejemplo, usando validación cruzada aleatoria en vez de TimeSeriesSplit), podría mostrar métricas artificialmente altas y generar falsas señales de confianza.

**Mitigación implementada:** Se utilizó TimeSeriesSplit y Walk-Forward Validation para garantizar que el modelo nunca vio el futuro durante el entrenamiento. Aun así, la brecha train-test es un recordatorio de que el overfitting es un riesgo permanente en modelos financieros.

# Conclusión General

A lo largo del proyecto se desarrolló un pipeline completo de Machine Learning para predecir la dirección futura del índice S&P 500. Se trabajó inicialmente con datos horarios y posteriormente con datos diarios, incorporando distintas etapas de preprocesamiento, ingeniería de características, validación temporal y optimización de hiperparámetros.

Se evaluaron múltiples enfoques, incluyendo Random Forest, Random Forest optimizado mediante Randomized Search CV y Grid Search CV, XGBoost y LightGBM. Asimismo, se incorporaron indicadores técnicos y variables externas relacionadas con volatilidad, tasas de interés, dólar y oro con el objetivo de mejorar la capacidad predictiva de los modelos.

Los resultados obtenidos mostraron mejoras moderadas en algunas métricas, especialmente tras la incorporación de nuevas variables predictoras. Sin embargo, ningún modelo logró alcanzar un nivel de desempeño significativamente superior al azar, obteniendo valores de AUC cercanos a 0,52. Esto evidencia la dificultad de anticipar los movimientos futuros del mercado utilizando únicamente información histórica y variables derivadas.

Como principal hallazgo, se observó que la ingeniería de características tuvo un impacto mayor que la optimización de hiperparámetros o el cambio de algoritmo. Además, la validación temporal y el análisis walk-forward permitieron comprobar que las señales encontradas no se mantuvieron de forma consistente a lo largo del tiempo.

En conclusión, el proyecto permitió implementar y evaluar de forma rigurosa distintas estrategias de modelado financiero, demostrando tanto el potencial como las limitaciones de las técnicas de Machine Learning aplicadas a la predicción de mercados bursátiles.
